# Predicting the expected number of kills given match interval, using Poisson
*What is the expected number of kills/deaths/assists within the next 10 minutes?* 

## Import Statements

In [ ]:
%%sql -r set_notebook_context
USE WAREHOUSE COMPUTE_WH;

USE DATABASE LEAGUE_RECORDS;

USE SCHEMA SILVER;

In [ ]:
from dataclasses import dataclass
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from snowflake.snowpark import DataFrame as SnowparkDataFrame

## Source Dataset Schema


In [ ]:
%%sql -r ranked_match_summary
WITH MATCH_MAX_INTERVAL AS (
    SELECT MATCH_ID, MAX(MINUTE) AS END_INTERVAL
    FROM PLAYER_INTERVAL_SILVER
    GROUP BY MATCH_ID
),
STATS_AT_END_INTERVAL AS (
    SELECT P.MATCH_ID,
        COALESCE(ROUND(
            AVG((P.CS + P.JUNGLE_CS) / NULLIF(MX.END_INTERVAL, 0))
        , 2), 0) AS AVG_CS_PER_MINUTE,
        COALESCE(ROUND(AVG(
            (P.KILLS + P.ASSISTS / (P.DEATHS + 1)) / NULLIF(MX.END_INTERVAL, 0)
        ), 2), 0) AS AVG_KDA_PER_MINUTE,
        COALESCE(ROUND(
            AVG(P.TOTAL_GOLD / NULLIF(MX.END_INTERVAL, 0)) 
        , 2), 0) AS AVG_GOLD_PER_MINUTE,
        SUM(P.TOTAL_GOLD) AS TOTAL_GOLDS,
        SUM(P.KILLS) AS TOTAL_KILLS,
        SUM(P.DEATHS) AS TOTAL_DEATHS,
        SUM(P.ASSISTS) AS TOTAL_ASSISTS
    FROM MATCH_MAX_INTERVAL AS MX
    JOIN PLAYER_INTERVAL_SILVER AS P
        ON P.MATCH_ID = MX.MATCH_ID
        AND P.MINUTE = MX.END_INTERVAL
    GROUP BY P.MATCH_ID
)
SELECT 
FROM MATCHES_SUMMARY_SILVER AS MS

SELECT 
    -- Context
    MS.MATCH_ID,
    ROUND(MS.GAME_DURATION / 60, 2) AS GAME_MINUTES,
    -- Response
    MS.AVERAGE_RANK,
    -- Features
    SX.AVG_CS_PER_MINUTE,
    SX.AVG_GOLD_PER_MINUTE,
    SX.AVG_KDA_PER_MINUTE,
    SX.TOTAL_GOLDS,
    SX.TOTAL_KILLS,
    SX.TOTAL_DEATHS,
    SX.TOTAL_ASSISTS
FROM STATS_AT_END_INTERVAL AS SX
JOIN MATCHES_SUMMARY_SILVER AS MS
    ON MS.MATCH_ID = SX.MATCH_ID
;

In [ ]:
def split_test_train(
    df: pd.DataFrame, 
    pct: float = 0.8,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not 0 < pct < 1:
        raise ValueError(f"pct must be between 0 and 1, got {pct}")

    train_set = df.sample(frac=pct, random_state=random_state)
    test_set = df.drop(train_set.index)

    print("----- SPLIT RESULTS -----")
    print(f"Base DF size --> {df.shape}")
    print(f"Training set --> {train_set.shape}")
    print(f"Testing set --> {test_set.shape}")
    print("-------------------------")

    return (train_set, test_set)

In [ ]:
def prior_prob_distribution(
    df: pd.DataFrame,
    group: str
) -> pd.DataFrame:
    return (df
        .reset_index(names="id")
        .groupby(group, as_index=False)
        .agg(p=(
            "id", 
            lambda x: x.count() / len(df)
        ))
        .sort_values("p", ascending=False)
    )

In [ ]:
def feature_dist_given_group(
    df: pd.DataFrame,
    feature: str,
    group_var: str, # Average Rank
    group_value: str, # Master, Grandmaster, etc.
    show_plots: bool = True
) -> tuple[float, float]:
    arr = (df
        .loc[df[group_var] == group_value, feature]
        .to_numpy()
    )
    arr_mean = np.mean(arr)
    arr_std = np.std(arr)
    
    if show_plots:
        fig, ax = plt.subplots()
        sns.histplot(arr, ax=ax)

        ax.set_title(f"Distribution of {feature} for {group_value} {group_var} matches.")
        ax.set_xlabel(feature)
        
        ax.axvline(arr_mean, color="black", linestyle="-", linewidth=1.5, label=f"mean = {arr_mean:.2f}")
        ax.axvline(arr_mean - arr_std, color="gray", linestyle="--", linewidth=1, label=f"std = {arr_std:.2f}")
        ax.axvline(arr_mean + arr_std, color="gray", linestyle="--", linewidth=1)
        
        plt.show()
        plt.close(fig)

    return arr_mean, arr_std

In [ ]:
feature_dist_given_group(
    df=ranked_match_summary.to_pandas(),
    feature="AVG_CS_PER_MINUTE",
    group_var="AVERAGE_RANK",
    group_value="Master",
    show_plots=True
)

In [ ]:
feature_dist_given_group(
    df=ranked_match_summary.to_pandas(),
    feature="TOTAL_DEATHS",
    group_var="AVERAGE_RANK",
    group_value="Master",
    show_plots=True
)

In [ ]:
def predict_many_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    response: str,
    features: list[str],
) -> pd.DataFrame:
    # ----- Prior from training data
    prior = prior_prob_distribution(df=train_df, group=response)
    print(prior)

    # ----- Per-rank, per-feature mean/std and log-prior, fit once on training data
    rank_params = {}
    for row in prior.itertuples():
        rank = getattr(row, response)
        p_prior = getattr(row, "p")

        feature_stats = {}
        for feature in features:
            mean, std = feature_dist_given_group(
                df=train_df,
                feature=feature,
                group_var=response,
                group_value=rank,
                show_plots=False,
            )
            feature_stats[feature] = {"mean": mean, "std": std}

        rank_params[rank] = {
            "log_prior": np.log(p_prior),
            "features": feature_stats,
        }

    # ----- Score every test row against every rank, summing log-likelihoods across features
    scores = pd.DataFrame(index=test_df.index)

    for rank, params in rank_params.items():
        rank_score = np.full(len(test_df), params["log_prior"])

        for feature in features:
            test_values = test_df[feature].to_numpy()
            stats = params["features"][feature]

            likelihood = norm.pdf(test_values, loc=stats["mean"], scale=stats["std"])
            likelihood = np.clip(likelihood, 1e-300, None)

            rank_score = rank_score + np.log(likelihood)

        scores[rank] = rank_score

    # ----- Pick the rank with the highest score per row
    predicted = scores.idxmax(axis=1)

    # ----- Assemble comparison dataframe
    result = test_df[features + [response]].copy()
    result["predicted"] = predicted.values
    result = result.rename(columns={response: "actual"})
    result["correct"] = result["actual"] == result["predicted"]

    return result

In [ ]:
HIGH_ELO_RANKS = {"Master", "Grandmaster", "Challenger"}

def add_elo_tier(df: pd.DataFrame, rank_col: str = "AVERAGE_RANK") -> pd.DataFrame:
    df = df.copy()
    df["ELO_TIER"] = df[rank_col].apply(
        lambda r: "High Elo" if r in HIGH_ELO_RANKS else "Low Elo"
    )
    return df

In [ ]:
def gaussian_nb(
    data: SnowparkDataFrame,
    response: str,
    features: list[str],
) -> pd.DataFrame:
    df = data.to_pandas()
    df = add_elo_tier(df, rank_col=response)

    train_df, test_df = split_test_train(
        df=df[features + ["ELO_TIER"]],
        pct=0.8,
        random_state=42,
    )

    results = predict_many_features(
        train_df=train_df,
        test_df=test_df,
        response="ELO_TIER",
        features=features,
    )

    accuracy = results["correct"].mean()
    print(f"Accuracy: {accuracy:.4f}")
    print(results.head(20))
    print("\nConfusion breakdown:")
    print(pd.crosstab(results["actual"], results["predicted"]))

    return results

In [ ]:
gaussian_nb(
    ranked_match_summary,
    response="AVERAGE_RANK",
    features=["AVG_CS_PER_MINUTE", "AVG_GOLD_PER_MINUTE", "AVG_KDA_PER_MINUTE"]
)